# 10 (heavy) - scDblFinder and decontX per sample

* 06a: split the raw object into one h5ad per sample on local disk (6.2 GB).
* 06b: scDblFinder (all cells, defaults, seed 12345) + default decontX (draft-QC cells, z = TotalVI Leiden r1), 46 samples, 12 in parallel.
* 06d: decontX strength ladder on 10 samples: default vs fixed delta (1,100), (1,300), (1,1000).
* 06e: final decontX delta (1,300), estimateDelta FALSE, for the 13 heavy in vivo cell samples.
R scDblFinder + celda through rpy2. Record: 06b ~20 min, 06d ~10 min, 06e ~25 min.

**Heavy notebook.** Nothing runs until you set `RUN_HEAVY = True` in the cell below. Long steps start in a detached
`screen` session with an exit sentinel (`WORK/logs/<name>.exit`); the wait cell polls the log. Re-run the light notebook named
below afterwards so that it uses these rebuilt outputs instead of the record's. Afterwards: notebooks 04 onwards.

**Runtime:** ~1 h, 12-13 parallel R workers
**Envs used (each step runs as a subprocess in its own env):** rapids-sc (06a), sc-preprocessing (06b, 06d, 06e)
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
RUN_HEAVY = False   # set True to run

In [ ]:
SPLIT = os.path.expanduser("~/liver_cross_species/repro_tmp_split"); G6 = rep("guilliams2022/06_doublets_ambient")
if RUN_HEAVY:   # nothing is touched unless the notebook is really run (review fix)
    guard(20, os.path.expanduser("~")); os.makedirs(SPLIT, exist_ok=True)
    link("guilliams2022/01_raw", "guilliams2022/02_qc", "guilliams2022/05_annotation_draft", "guilliams2022/03_integration")
    fresh("guilliams2022/06_doublets_ambient/per_sample", "guilliams2022/06_doublets_ambient/ladder", "guilliams2022/06_doublets_ambient/final_dx")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06a_split_samples.py
"""Step 06a - split the raw Fig 1B object (all 389,056 cells) into one h5ad per sample on local disk for the
scDblFinder/decontX workers. obs carries keep (draft QC), scrublet_doublet and leiden_totalvi_r1 (kept cells only).
Usage: 06a_split_samples.py <OUT> <SPLITDIR>"""
import os
import sys

import anndata as ad
import pandas as pd

OUT, SPLIT = sys.argv[1], sys.argv[2]
os.makedirs(SPLIT, exist_ok=True)
a = ad.read_h5ad(f"{OUT}/01_raw/guilliams2022_fig1b_raw.h5ad")
f = pd.read_parquet(f"{OUT}/02_qc/guilliams2022_fig1b_qc_flags.parquet")
assert (f.index == a.obs_names).all()
E = pd.read_parquet(f"{OUT}/05_annotation_draft/cells_annotation_draft.parquet")
a.obs["keep"] = f["keep"].values
a.obs["scrublet_doublet"] = f["scrublet_doublet"].values
a.obs["scrublet_score"] = f["doublet_score"].values
a.obs["z"] = E["leiden_totalvi_r1"].reindex(a.obs_names).astype(str).values
a.obs["annotation_draft"] = E["annotation_draft"].reindex(a.obs_names).astype(str).values
assert (a.obs.loc[a.obs.keep, "z"] != "nan").all()
a.obsm.clear()
for s, idx in a.obs.groupby("sample", observed=True).indices.items():
    a[idx].copy().write_h5ad(f"{SPLIT}/{s}.h5ad")
    print(s, len(idx), flush=True)

In [ ]:
if RUN_HEAVY: run("06a_split_samples.py", "rapids-sc", args=[rep("guilliams2022"), SPLIT])

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06b_doublets_decontx_sample.py
"""Step 06b - one sample: scDblFinder on all cells (MASLD doublets_v6 settings: defaults, raw counts, seed 12345), then
decontX on the draft-QC kept cells (MASLD fq_08 settings: z = clusters, varGenes 5000, seed 12345; here z = TotalVI Leiden r1
and decontX defaults: delta c(10,10), estimateDelta TRUE - no ambient tiers exist for this dataset yet).
sc-preprocessing env (R scDblFinder + celda via rpy2).  Usage: 06b_doublets_decontx_sample.py <SPLITDIR> <RESDIR> <sample>
Writes <RESDIR>/<sample>.cells.parquet (scDblFinder score/class, decontX contamination) and <sample>.decontx.npz (kept cells)."""
import os
import sys
import time
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.simplefilter("ignore")
SPLIT, RES, S = sys.argv[1:4]
SEED = 12345
os.environ.setdefault("R_HOME", f"{sys.prefix}/lib/R")
import rpy2.robjects as ro  # noqa: E402
from rpy2.robjects import numpy2ri  # noqa: E402

ro.r("suppressPackageStartupMessages({library(scDblFinder); library(SingleCellExperiment); library(celda); library(Matrix)})")
CONV = ro.default_converter + numpy2ri.converter


def push(m, name):
    m = sp.csc_matrix(m)
    m.sort_indices()
    with CONV.context():
        ro.globalenv["._i"] = m.indices.astype(np.int32)
        ro.globalenv["._p"] = m.indptr.astype(np.int32)
        ro.globalenv["._x"] = m.data.astype(np.float64)
        ro.globalenv["._d"] = np.asarray(m.shape, dtype=np.int32)
    ro.r(f'{name} <- new("dgCMatrix", i=as.integer(._i), p=as.integer(._p), x=as.numeric(._x), Dim=as.integer(._d))')


def pull(name, shape):
    with CONV.context():
        i = np.asarray(ro.r(f"{name}@i")); p = np.asarray(ro.r(f"{name}@p")); x = np.asarray(ro.r(f"{name}@x"))
    return sp.csc_matrix((x.astype(np.float32), i, p), shape=shape)


t0 = time.time()
a = ad.read_h5ad(f"{SPLIT}/{S}.h5ad")
X = sp.csr_matrix(a.X)
assert np.array_equal(X.data, np.round(X.data)), "not integer counts"
res = pd.DataFrame(index=a.obs_names, data={"scDblFinder_score": np.nan, "scDblFinder_class": "not_run",
                                            "decontX_contamination": np.nan})
push(X.T, "m")
ro.r(f"set.seed({SEED}); sce <- scDblFinder(SingleCellExperiment(list(counts = m)), verbose = FALSE); "
     "dsc <- as.numeric(sce$scDblFinder.score); dcl <- as.character(sce$scDblFinder.class); rm(sce, m); invisible(gc())")
res["scDblFinder_score"] = np.asarray(ro.r("dsc"))
res["scDblFinder_class"] = list(ro.r("dcl"))
t1 = time.time()

k = np.flatnonzero(a.obs["keep"].values)
sub = X[k]
gk = np.flatnonzero(np.asarray((sub > 0).sum(0)).ravel() > 0)
push(sub[:, gk].T, "m")
z = a.obs["z"].values[k].astype(str)
with CONV.context():
    ro.globalenv["zl"] = np.asarray(z, dtype=str)
zarg = "as.character(zl)" if len(set(z)) > 1 else "NULL"
ro.r(f"res <- decontX(m, z = {zarg}, varGenes = 5000, seed = {SEED}, verbose = FALSE); "
     "dc <- drop0(round(res$decontXcounts)); cont <- res$contamination; est <- res$estimates; rm(res)")
cg = pull("dc", (len(gk), len(k))).T.tocsr()
corr = sp.csr_matrix((cg.data, gk[cg.indices], cg.indptr), shape=sub.shape, dtype=np.float32)
res.iloc[k, 2] = np.asarray(ro.r("cont"))
os.makedirs(RES, exist_ok=True)
sp.save_npz(f"{RES}/{S}.decontx.npz", corr)
res.to_parquet(f"{RES}/{S}.cells.parquet")
print(f"{S}: {len(res)} cells, scDblFinder doublets {(res.scDblFinder_class == 'doublet').mean():.3f} ({t1 - t0:.0f}s); "
      f"decontX on {len(k)}: median contamination {np.nanmedian(res.decontX_contamination):.3f}, "
      f"UMIs kept {corr.sum() / sub.sum():.3f} ({time.time() - t1:.0f}s)", flush=True)

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06d_decontx_ladder.py
"""Step 06d - decontX strength ladder on one sample (MASLD decontx_titration idea): default (estimated delta) vs fixed priors
(1,100), (1,300), (1,1000), estimateDelta FALSE; z = TotalVI Leiden r1; varGenes 5000; seed 12345.
Metrics: Alb removed from non-epithelial draft types, Alb kept in hepatocytes, UMIs kept, cells < 200 UMI.
Usage: 06d_decontx_ladder.py <SPLITDIR> <RESDIR> <sample>  -> <RESDIR>/<sample>.ladder.tsv"""
import os
import sys
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.simplefilter("ignore")
SPLIT, RES, S = sys.argv[1:4]
os.environ.setdefault("R_HOME", f"{sys.prefix}/lib/R")
import rpy2.robjects as ro  # noqa: E402
from rpy2.robjects import numpy2ri  # noqa: E402

ro.r("suppressPackageStartupMessages({library(celda); library(Matrix)})")
CONV = ro.default_converter + numpy2ri.converter
a = ad.read_h5ad(f"{SPLIT}/{S}.h5ad")
a = a[a.obs["keep"].values].copy()
X = sp.csr_matrix(a.X)
gk = np.flatnonzero(np.asarray((X > 0).sum(0)).ravel() > 0)
m = sp.csc_matrix(X[:, gk].T); m.sort_indices()
with CONV.context():
    ro.globalenv["._i"] = m.indices.astype(np.int32); ro.globalenv["._p"] = m.indptr.astype(np.int32)
    ro.globalenv["._x"] = m.data.astype(np.float64); ro.globalenv["._d"] = np.asarray(m.shape, dtype=np.int32)
    ro.globalenv["zl"] = a.obs["z"].astype(str).values
ro.r('m <- new("dgCMatrix", i=as.integer(._i), p=as.integer(._p), x=as.numeric(._x), Dim=as.integer(._d))')
alb = np.flatnonzero(a.var["symbol"].astype(str).values[gk] == "Alb")[0]
typ = a.obs["annotation_draft"].astype(str).values
hep, nonepi = typ == "Hepatocyte", ~np.isin(typ, ["Hepatocyte", "Cholangiocyte"])
a0 = np.asarray(X[:, gk[alb]].todense()).ravel(); u0 = np.asarray(X.sum(1)).ravel()
rows = []
for name, arg in [("default_est", ""), ("1_100", ", delta = c(1, 100), estimateDelta = FALSE"),
                  ("1_300", ", delta = c(1, 300), estimateDelta = FALSE"), ("1_1000", ", delta = c(1, 1000), estimateDelta = FALSE")]:
    ro.r(f"res <- decontX(m, z = as.character(zl), varGenes = 5000, seed = 12345, verbose = FALSE{arg}); "
         f"dc <- res$decontXcounts; cont <- res$contamination; ab <- as.numeric(dc[{alb + 1}, ]); us <- colSums(dc)")
    a1 = np.asarray(ro.r("ab")); u1 = np.asarray(ro.r("us")); cont = np.asarray(ro.r("cont"))
    rows.append(dict(sample=S, setting=name, n=len(a1), median_contamination=np.median(cont),
                     nonepi_Alb_cp10k_raw=(a0[nonepi] / np.maximum(u0[nonepi], 1) * 1e4).mean(),
                     nonepi_Alb_cp10k_dx=(a1[nonepi] / np.maximum(u1[nonepi], 1) * 1e4).mean(),
                     nonepi_Alb_removed=1 - a1[nonepi].sum() / max(a0[nonepi].sum(), 1),
                     hep_Alb_kept=a1[hep].sum() / a0[hep].sum() if a0[hep].sum() else np.nan,
                     hep_umis_kept=u1[hep].sum() / u0[hep].sum() if hep.any() else np.nan,
                     nonepi_umis_kept=u1[nonepi].sum() / u0[nonepi].sum(), lt200=int((u1 < 200).sum())))
    print(rows[-1], flush=True)
os.makedirs(RES, exist_ok=True)
pd.DataFrame(rows).to_csv(f"{RES}/{S}.ladder.tsv", sep="\t", index=False, float_format="%.4g")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06e_decontx_final_sample.py
"""Step 06e - final decontX for one tier-1 sample: fixed prior delta = (1, 300), estimateDelta FALSE, z = TotalVI Leiden r1,
varGenes 5000, seed 12345 (chosen from the 06d ladder, user go 2026-11-11). Draft-QC kept cells only.
Usage: 06e_decontx_final_sample.py <SPLITDIR> <RESDIR> <sample>  -> <RESDIR>/<sample>.dx300.npz, <sample>.dx300.parquet"""
import os
import sys
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.simplefilter("ignore")
SPLIT, RES, S = sys.argv[1:4]
os.environ.setdefault("R_HOME", f"{sys.prefix}/lib/R")
import rpy2.robjects as ro  # noqa: E402
from rpy2.robjects import numpy2ri  # noqa: E402

ro.r("suppressPackageStartupMessages({library(celda); library(Matrix)})")
CONV = ro.default_converter + numpy2ri.converter
a = ad.read_h5ad(f"{SPLIT}/{S}.h5ad")
a = a[a.obs["keep"].values].copy()
X = sp.csr_matrix(a.X)
gk = np.flatnonzero(np.asarray((X > 0).sum(0)).ravel() > 0)
m = sp.csc_matrix(X[:, gk].T); m.sort_indices()
with CONV.context():
    ro.globalenv["._i"] = m.indices.astype(np.int32); ro.globalenv["._p"] = m.indptr.astype(np.int32)
    ro.globalenv["._x"] = m.data.astype(np.float64); ro.globalenv["._d"] = np.asarray(m.shape, dtype=np.int32)
    ro.globalenv["zl"] = a.obs["z"].astype(str).values
ro.r('m <- new("dgCMatrix", i=as.integer(._i), p=as.integer(._p), x=as.numeric(._x), Dim=as.integer(._d))')
ro.r("res <- decontX(m, z = as.character(zl), delta = c(1, 300), estimateDelta = FALSE, varGenes = 5000, seed = 12345, "
     "verbose = FALSE); dc <- drop0(round(res$decontXcounts)); cont <- res$contamination; rm(res)")
with CONV.context():
    i = np.asarray(ro.r("dc@i")); p = np.asarray(ro.r("dc@p")); x = np.asarray(ro.r("dc@x"))
cg = sp.csc_matrix((x.astype(np.float32), i, p), shape=(len(gk), a.n_obs)).T.tocsr()
corr = sp.csr_matrix((cg.data, gk[cg.indices], cg.indptr), shape=X.shape, dtype=np.float32)
sp.save_npz(f"{RES}/{S}.dx300.npz", corr)
pd.DataFrame({"decontX_contamination_final": np.asarray(ro.r("cont"))}, index=a.obs_names).to_parquet(f"{RES}/{S}.dx300.parquet")
print(S, a.n_obs, f"UMIs kept {corr.sum() / X.sum():.3f}", flush=True)

In [ ]:
P = PY["sc-preprocessing"]; T = "export OMP_NUM_THREADS=2 OPENBLAS_NUM_THREADS=2 MKL_NUM_THREADS=2"
LADDER = "CS140 CS139 CS93 CS144 CS89 CS115 ABU19 ABU20 ABU21 CS98"
TIER1 = "CS114 CS137 CS138 CS139 CS144 CS48 CS52 CS88 CS89 CS92 CS93 CS96 CS97"
launch("dbl_dx", [T, f"mkdir -p {LOGS}/06b {LOGS}/06d {LOGS}/06e",
    f"ls -S {SPLIT}/*.h5ad | xargs -n1 basename | sed 's/\\.h5ad$//' | xargs -P 12 -I{{}} sh -c '{P} -I {WORK}/06b_doublets_decontx_sample.py {SPLIT} {G6}/per_sample {{}} > {LOGS}/06b/{{}}.log 2>&1'",
    f"echo {LADDER} | tr ' ' '\\n' | xargs -P 10 -I{{}} sh -c '{P} -I {WORK}/06d_decontx_ladder.py {SPLIT} {G6}/ladder {{}} > {LOGS}/06d/{{}}.log 2>&1'",
    f"echo {TIER1} | tr ' ' '\\n' | xargs -P 13 -I{{}} sh -c '{P} -I {WORK}/06e_decontx_final_sample.py {SPLIT} {G6}/final_dx {{}} > {LOGS}/06e/{{}}.log 2>&1'",
    f"[ $(ls {G6}/per_sample/*.cells.parquet | wc -l) -eq 46 ] && [ $(ls {G6}/final_dx/*.dx300.npz | wc -l) -eq 13 ]"])
wait_for("dbl_dx")

Delete the split folder (`~/liver_cross_species/repro_tmp_split`, 6.2 GB) when notebook 04 has been re-run.